# 04 - Run Security System

## 24/7 Face Security Monitoring System

Complete security system with:

1. 🔴 24/7 Monitoring Mode
2. 👤 Face Detection & Recognition
3. 🚨 Alert System (Telegram + Desktop)
4. 📹 Live Video Display
5. 📊 System Statistics

---

**⚠️ Important:** This notebook runs a continuous monitoring loop. Press 'q' in the video window to stop.

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import time
from datetime import datetime
from src.face_detector import FaceDetector
from src.face_recognizer import FaceRecognizer
from src.database_manager import DatabaseManager
from src.alert_system import AlertSystem
from src.camera_handler import CameraHandler
from utils.logger import setup_logging
from config import *

print("Face Security Alert System")
print("="*60)

## 1. Initialize System Components

In [ ]:
# Setup logging
logger = setup_logging(
    log_dir=LOGGING_CONFIG['log_dir'],
    log_level="INFO",
    log_to_file=True,
    log_to_console=True
)

# Initialize components
print("Initializing components...")

face_detector = FaceDetector(
    backend=DETECTION_CONFIG['backend'],
    min_face_size=DETECTION_CONFIG['min_face_size']
)

face_recognizer = FaceRecognizer(
    model_name=RECOGNITION_CONFIG['model'],
    distance_metric=RECOGNITION_CONFIG['distance_metric'],
    threshold=RECOGNITION_CONFIG['threshold'],
    cooldown_seconds=ALERT_CONFIG['cooldown_seconds'],
    detection_threshold_seconds=ALERT_CONFIG['detection_threshold_seconds']
)

db_manager = DatabaseManager(
    known_faces_dir=DATABASE_CONFIG['known_faces_dir'],
    encodings_file=DATABASE_CONFIG['encodings_file'],
    min_images_per_person=DATABASE_CONFIG['min_images_per_person'],
    max_images_per_person=DATABASE_CONFIG['max_images_per_person']
)

alert_system = AlertSystem(
    telegram_bot_token=TELEGRAM_CONFIG['bot_token'],
    telegram_chat_id=TELEGRAM_CONFIG['chat_id'],
    enable_telegram=ALERT_CONFIG['enable_telegram'],
    enable_desktop=ALERT_CONFIG['enable_desktop'],
    save_unknown_faces=ALERT_CONFIG['save_unknown_faces'],
    unknown_faces_dir=ALERT_CONFIG['unknown_faces_dir']
)

camera_handler = CameraHandler(
    camera_source=CAMERA_CONFIG['source'],
    width=CAMERA_CONFIG['width'],
    height=CAMERA_CONFIG['height'],
    show_video=True,
    show_fps=True
)

print("✅ All components initialized")

## 2. Load Face Database

In [ ]:
database = db_manager.load_database()

if database:
    stats = db_manager.get_database_stats()
    print(f"✅ Database loaded: {stats['num_people']} people")
    for name in database.keys():
        print(f"   - {name}")
else:
    print("❌ Database is empty")
    print("   Run notebooks/02_build_face_database.ipynb first")

## 3. Test Alert System

In [ ]:
print("Testing alert system...\n")

# Test desktop notification
if ALERT_CONFIG['enable_desktop']:
    if alert_system.test_desktop_notification():
        print("✅ Desktop notifications working")
    else:
        print("❌ Desktop notifications failed")

# Test Telegram
if ALERT_CONFIG['enable_telegram']:
    if alert_system.test_telegram_connection():
        print("✅ Telegram connection working")
    else:
        print("❌ Telegram connection failed")
        print("   Check your .env file configuration")

## 4. Start Security Monitoring

**Press 'q' in the video window to stop monitoring.**

In [ ]:
def run_security_monitoring():
    """Run the complete security monitoring system."""
    
    if not database:
        print("❌ Cannot start: Database is empty")
        return
    
    # Open camera
    if not camera_handler.open():
        print("❌ Failed to open camera")
        return
    
    print("\n" + "="*60)
    print("🔴 SECURITY MONITORING ACTIVE")
    print("="*60)
    print(f"Telegram alerts: {'ON' if ALERT_CONFIG['enable_telegram'] else 'OFF'}")
    print(f"Desktop alerts: {'ON' if ALERT_CONFIG['enable_desktop'] else 'OFF'}")
    print(f"Alert cooldown: {ALERT_CONFIG['cooldown_seconds']}s")
    print(f"Detection threshold: {ALERT_CONFIG['detection_threshold_seconds']}s")
    print("\nPress 'q' to quit, 's' for screenshot")
    print("="*60 + "\n")
    
    frame_count = 0
    start_time = time.time()
    
    try:
        while True:
            ret, frame = camera_handler.read_frame()
            if not ret:
                break
            
            # Process every Nth frame
            if frame_count % DETECTION_CONFIG['process_every_n_frames'] == 0:
                # Detect faces
                faces = face_detector.detect_faces(frame)
                
                for face_data in faces:
                    x, y, w, h = face_detector.get_face_coordinates(face_data)
                    face_img = face_detector.extract_face_region(frame, face_data)
                    
                    if face_img is not None:
                        # Recognize
                        name, confidence, distance = face_recognizer.recognize_face(
                            face_img, database
                        )
                        
                        if name:
                            # Known face - green box
                            face_detector.draw_face_box(
                                frame, x, y, w, h,
                                color=(0, 255, 0),
                                label=name,
                                confidence=confidence
                            )
                        else:
                            # Unknown face - red box
                            face_detector.draw_face_box(
                                frame, x, y, w, h,
                                color=(0, 0, 255),
                                label="Unknown",
                                confidence=confidence
                            )
                            
                            # Track and potentially send alert
                            person_id = f"unknown_{x}_{y}"
                            if face_recognizer.update_detection_tracking(person_id):
                                if face_recognizer.should_send_alert(person_id):
                                    print(f"🚨 ALERT: Unknown person detected at {datetime.now()}")
                                    alert_system.send_alert(
                                        face_image=face_img,
                                        full_frame=frame.copy(),
                                        confidence=confidence
                                    )
            
            frame_count += 1
            
            # Add overlays
            frame = camera_handler.draw_fps(frame)
            frame = camera_handler.draw_timestamp(frame)
            frame = camera_handler.draw_status_message(
                frame, "🔴 MONITORING", "success"
            )
            
            # Display
            key = camera_handler.display_frame(frame)
            
            if key == ord('q'):
                print("\nStopping monitoring...")
                break
            elif key == ord('s'):
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                screenshot_path = f"../logs/screenshot_{timestamp}.jpg"
                cv2.imwrite(screenshot_path, frame)
                print(f"Screenshot saved: {screenshot_path}")
            
            # Periodic cleanup
            if frame_count % 1000 == 0:
                face_recognizer.clear_old_tracking_data()
        
    except KeyboardInterrupt:
        print("\nInterrupted by user")
    finally:
        camera_handler.release()
        elapsed = time.time() - start_time
        print(f"\nMonitoring duration: {elapsed/60:.1f} minutes")
        print(f"Total frames processed: {frame_count}")
        print("\n✅ System stopped")

# Start monitoring
run_security_monitoring()

## 5. View Alert Statistics

In [ ]:
stats = alert_system.get_alert_stats()

print("Alert Statistics")
print("="*60)
print(f"Total unknown faces saved: {stats['total_photos']}")
print(f"Storage location: {stats['storage_path']}")

if stats['photos']:
    print("\nRecent alerts:")
    for photo in stats['photos'][:5]:
        print(f"  - {photo}")
else:
    print("\nNo alerts recorded yet")

## Summary

### ✅ Security System Features

- ✅ 24/7 monitoring capability
- ✅ Real-time face detection
- ✅ Face recognition with confidence scores
- ✅ Alert system (Telegram + Desktop)
- ✅ Unknown face photo capture
- ✅ Alert cooldown to prevent spam
- ✅ Live video display with overlays

### 💡 Tips

- Adjust `RECOGNITION_CONFIG['threshold']` in config.py for accuracy
- Configure Telegram in .env for remote alerts
- Check logs/ directory for system logs
- Review data/unknown_faces/ for alerts

### 📚 For Production Use

Use the command-line script for 24/7 operation:
```bash
python scripts/run_security_system.py
```

---

**🎉 Setup Complete!** Your Face Security Alert System is ready.